In [13]:
import pandas as pd
from collections import Counter
import spacy

nlp = spacy.load("en_core_web_sm")

def build_vocab(csv_path, min_freq=2):
    df = pd.read_csv(csv_path)

    # CLEAN THE COLUMN HERE
    df["Comment"] = df["Comment"].fillna("").astype(str)

    counter = Counter()

    for text in df["Comment"]:
        # safety: ensure text is string
        if not isinstance(text, str):
            text = ""

        doc = nlp(text)
        tokens = [t.text.lower() for t in doc]
        counter.update(tokens)

    vocab = {"<pad>": 0, "<unk>": 1}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

class YouTubeCommentsDataset(Dataset):
    def __init__(self, csv_path, vocab, max_len=128, device: torch.device = torch.device("cpu")):
        self.df = pd.read_csv(csv_path)
        self.device = device

        # clean Comment column
        self.df["Comment"] = self.df["Comment"].fillna("").astype(str)

        # map text labels → integers
        label_map = {"negative": 0, "neutral": 1, "positive": 2}
        self.df["Sentiment"] = (
            self.df["Sentiment"]
            .fillna("neutral")          # fallback
            .str.lower()                # normalize
            .map(label_map)             # convert to int
            .astype(int)
        )

        self.vocab = vocab
        self.max_len = max_len
        self.pad_id = vocab["<pad>"]
        self.unk_id = vocab["<unk>"]

    def encode(self, text):
        doc = nlp(text)
        tokens = [t.text.lower() for t in doc]
        ids = [self.vocab.get(tok, self.unk_id) for tok in tokens][:self.max_len]
        ids += [self.pad_id] * (self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long, device=self.device)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = self.encode(row["Comment"])
        y = torch.tensor(row["Sentiment"], dtype=torch.long, device=self.device)
        return x, y


In [15]:
import torch
import torch.nn as nn

class SimpleSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        out, (h, c) = self.encoder(emb)
        logits = self.classifier(h[-1])
        return logits


In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

print(f"Device used: {device.type}")

csv_path = "data/YoutubeCommentsDataSet.csv"

# build vocab
vocab = build_vocab(csv_path)

# dataset + loader
dataset = YouTubeCommentsDataset(csv_path, vocab, device=device)
loader = DataLoader(dataset, batch_size=(128 if device == "cuda" else 32), shuffle=True, generator=torch.Generator("cuda" if torch.cuda.is_available() else "cpu"))

# model
model = SimpleSentiment(vocab_size=len(vocab))
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.to(device)

# training
for epoch in range(19):
    losses = []
    all_preds = []
    all_labels = []

    for x, y in loader:
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        all_preds.extend(logits.argmax(dim=1).tolist())
        all_labels.extend(y.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"Epoch {epoch+1} | Loss={sum(losses)/len(losses):.4f} | Acc={acc:.3f} | F1={f1:.3f}")

# save model
torch.save({"model": model.state_dict(), "vocab": vocab}, "sentiment_model.pt")


Device used: cuda
Epoch 1 | Loss=0.9016 | Acc=0.624 | F1=0.276
Epoch 2 | Loss=0.7688 | Acc=0.676 | F1=0.441
Epoch 3 | Loss=0.6183 | Acc=0.735 | F1=0.534
Epoch 4 | Loss=0.5239 | Acc=0.776 | F1=0.629
Epoch 5 | Loss=0.4416 | Acc=0.814 | F1=0.711
Epoch 6 | Loss=0.3560 | Acc=0.856 | F1=0.775
Epoch 7 | Loss=0.2760 | Acc=0.891 | F1=0.828
Epoch 8 | Loss=0.2105 | Acc=0.922 | F1=0.875
Epoch 9 | Loss=0.1551 | Acc=0.947 | F1=0.917
Epoch 10 | Loss=0.1187 | Acc=0.961 | F1=0.940
Epoch 11 | Loss=0.0873 | Acc=0.972 | F1=0.958
Epoch 12 | Loss=0.0697 | Acc=0.979 | F1=0.969
Epoch 13 | Loss=0.0553 | Acc=0.983 | F1=0.975
Epoch 14 | Loss=0.0398 | Acc=0.989 | F1=0.985
Epoch 15 | Loss=0.0355 | Acc=0.990 | F1=0.986
Epoch 16 | Loss=0.0386 | Acc=0.989 | F1=0.986
Epoch 17 | Loss=0.0312 | Acc=0.991 | F1=0.988
Epoch 18 | Loss=0.0265 | Acc=0.992 | F1=0.990
Epoch 19 | Loss=0.0206 | Acc=0.994 | F1=0.992
Epoch 20 | Loss=0.0258 | Acc=0.992 | F1=0.989
